In [ ]:
"""
Движење во матрица — алгоритми за пребарување на патека
=========================================================

Овој фајл ги содржи сите алгоритми и помошни функции од нотбукот
"05_Движење_во_матрица.ipynb", извлечени и средени во чист .py фајл,
со подробни објаснувања (на македонски) КОГА и ЗОШТО се користи секоја
функција.

Содржина:
    1. Поставки на матрицата и ѕидови (WALLS)
    2. "Планина" (тежински терен) и функција за тежина на чекор
    3. Функции за проширување на состојби (соседи): 4-смерно и 8-смерно
    4. Хеуристики за растојание: Манхетен, Чебишев, Евклид
    5. Алгоритми за пребарување:
        - breadth_first_search (BFS)
        - depth_first_search (DFS)
        - uniform_cost_search (UCS)
        - gready_search (Greedy best-first)
        - a_star_search (A*)
    6. Кратко резиме / водич "кога која функција да се користи"
    7. Пример за употреба (if __name__ == "__main__")

Забелешка: Графичкиот/интерактивен дел од нотбукот (Plotly + ipywidgets
Simulation класата) е намерно изоставен бидејќи зависи од Jupyter
околина. Овде наместо тоа имаме едноставна конзолна демонстрација која
работи во секој обичен Python фајл/терминал.
"""

import math
import heapq
from collections import deque


# ---------------------------------------------------------------------------
# 1. Поставки на матрицата (мрежата) и ѕидовите (пречките)
# ---------------------------------------------------------------------------

ROWS, COLUMNS = 20, 30

# Секој ѕид е правоаголник дефиниран преку (x_min, x_max) и (y_min, y_max).
# Се користат при проверка дали едно поле е "проодно" (is_valid_square).
WALLS = [
    {'x': (12.5, 23.5), 'y': (7.5, 9.5)},
    {'x': (8.5, 10.5), 'y': (2.5, 13.5)},
    {'x': (16.5, 19.5), 'y': (12.5, 16.5)},
]


# ---------------------------------------------------------------------------
# 2. "Планина" — тежински терен (опционално, само за тежинските пребарувања)
# ---------------------------------------------------------------------------

def create_mountain():
    """
    Генерира матрица MOUNTAIN со "височини" во облик на планина
    (параболична функција со врв во десната половина на мрежата).

    КОГА да се користи:
        - Само ако сакаш терен со различна "тешкотија" на движење
          (пр. движење нагоре по планина е поскапо отколу movewise по равен терен).
        - Доколку ти треба чисто униформна мрежа без тежини, оваа функција
          воопшто не е потребна — само игнорирај ја и користи get_weight
          со константна тежина 1.

    Враќа:
        mountain        - 2D листа (ROWS x COLUMNS) со [висина, висина, висина]
                           за секое поле (форматот со 3 еднакви вредности
                           потекнува од Plotly go.Image кој очекува RGB).
        mountain_image  - Plotly go.Image објект (само за визуелизација,
                           не е потребен ако нема да цртаме).
    """
    def f(x, y):
        return 1 - (x - 2 * COLUMNS // 3) ** 2 - (y - ROWS // 2) ** 2

    mountain = [[f(x, y) for x in range(COLUMNS)] for y in range(ROWS)]
    min_value = min(min(mountain))
    for y in range(ROWS):
        for x in range(COLUMNS):
            mountain[y][x] = [mountain[y][x] - min_value] * 3

    mountain_image = None
    try:
        from plotly import graph_objects as go
        min_value2 = min(min(min(mountain)))
        max_value2 = max(max(max(mountain)))
        mountain_image = go.Image(z=mountain, zmin=[min_value2] * 4, zmax=[max_value2] * 4)
    except ImportError:
        # Plotly не е задолжителен за самата логика на пребарувањето.
        pass

    return mountain, mountain_image


def get_weight(source_state, destination_state, mountain):
    """
    Враќа колку е "тешко" (колкава е цената) да се помине од source_state
    кон destination_state, во зависност од разликата во височина во MOUNTAIN.

    КОГА да се користи:
        - Единствено кај тежинските алгоритми: uniform_cost_search и
          a_star_search. Кај breadth_first_search, depth_first_search и
          gready_search не се користи цена/тежина на чекор (тие не градат
          оптимална по тежина патека во класична смисла — greedy и DFS/BFS
          гледаат само структура/хеуристика, не реална цена).
        - Ако немаш терен со различни тежини (само празна мрежа), можеш да
          вратиш константно 1 за секој чекор наместо ова.

    Логика: качување нагоре (поголема височина на дестинацијата) е поскапо,
    спуштање надолу не е поевтино од минимум 1 (секој чекор кошта барем 1).
    """
    source_x, source_y = source_state
    destination_x, destination_y = destination_state
    weight = mountain[destination_y][destination_x][0] - mountain[source_y][source_x][0]
    return 1 + max(weight, 0)


# ---------------------------------------------------------------------------
# 3. Функции за проширување на состојби (генерирање соседи)
# ---------------------------------------------------------------------------

def is_valid_square(square):
    """
    Проверува дали полето (квадратчето) е во границите на мрежата и дали
    не паѓа во некој од ѕидовите (WALLS).

    КОГА да се користи:
        - Секогаш кога генерираш соседи на едно поле (внатре во
          expand_state_4 и expand_state_8). Не се користи самостојно
          надвор од контекст на пребарување, освен ако рачно тестираш
          дали една координата е валидна.
    """
    x, y = square
    if not (0 <= x < COLUMNS) or not (0 <= y < ROWS):
        return False
    for wall in WALLS:
        if wall['x'][0] < x < wall['x'][1] and wall['y'][0] < y < wall['y'][1]:
            return False
    return True


def expand_state_4(state):
    """
    Генерира соседи на 'state' само во 4 насоки: лево, десно, горе, долу
    (без дијагонали).

    КОГА да се користи:
        - Кога движењето е дозволено само хоризонтално/вертикално
          (типичен "градски" / "taxicab" модел на движење).
        - Совршено се комбинира со manhattan_distance како хеуристика,
          бидејќи Манхетен растојание е точно минималниот број чекори
          во 4-смерна мрежа без пречки.
    """
    next_states = []
    x, y = state
    neighbour_states = [(x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)]
    for neighbour_state in neighbour_states:
        if is_valid_square(neighbour_state):
            next_states.append(neighbour_state)
    return next_states


def expand_state_8(state):
    """
    Генерира соседи на 'state' во 8 насоки (вклучувајќи и 4 дијагонали).

    КОГА да се користи:
        - Кога дозволуваш и дијагонално движење (поприродно/побрзо
          движење, чести во игри и роботика).
        - Се комбинира најприродно со chebyshev_distance (Чебишев е точниот
          минимален број чекори во 8-смерна мрежа без пречки) или со
          eucledian_distance (ако сакаш "права линија" проценка).
    """
    next_states = []
    x, y = state
    neighbour_states = [
        (x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1),
        (x - 1, y - 1), (x - 1, y + 1), (x + 1, y - 1), (x + 1, y + 1),
    ]
    for neighbour_state in neighbour_states:
        if is_valid_square(neighbour_state):
            next_states.append(neighbour_state)
    return next_states


# ---------------------------------------------------------------------------
# 4. Хеуристики / функции за растојание
# ---------------------------------------------------------------------------

def manhattan_distance(state_1, state_2):
    """
    Манхетен (Taxicab) растојание: |dx| + |dy|.

    КОГА да се користи:
        - Кога движењето е ограничено на 4 насоки (expand_state_4).
        - Дава точна (admissible и consistent) проценка во таков случај,
          па е добра хеуристика за gready_search и a_star_search.
        - Ако се користи со 8-смерно движење, ХЕУРИСТИКАТА ЌЕ
          ПРЕЦЕНУВА (overestimate) реалната цена, што кај A* може да
          ја нарушипи оптималноста (хеуристиката треба да биде
          admissible, т.е. да не преценува).
    """
    return abs(state_1[0] - state_2[0]) + abs(state_1[1] - state_2[1])


def chebyshev_distance(state_1, state_2):
    """
    Чебишев (Chessboard / King-move) растојание: max(|dx|, |dy|).

    КОГА да се користи:
        - Кога движењето е дозволено во 8 насоки (expand_state_8), бидејќи
          дијагоналниот чекор и хоризонтален/вертикален чекор имаат
          еднаква "цена" (1 потег) - токму како движењето на крал во шах.
        - Ова е точна (admissible) хеуристика за 8-смерна мрежа без
          дополнителни тежини на терена, па е добар избор за A* и Greedy.
    """
    return max(abs(state_1[0] - state_2[0]), abs(state_1[1] - state_2[1]))


def eucledian_distance(state_1, state_2):
    """
    Евклидово (право-линиско) растојание: sqrt(dx^2 + dy^2).

    КОГА да се користи:
        - Кога движењето е "слободно" (континуирано, не само по решетка)
          или сакаш реалистична проценка на воздушна линија.
        - Кај 8-смерна решеткаста мрежа Евклидовото растојание е помало
          или еднакво на реалната цена (секогаш admissible), но е
          "послаба"/помалку информативна хеуристика од Чебишев бидејќи
          подоцена ја потценува цената повеќе → A* со Евклид типично
          проширува повеќе јазли (поспоро, но сепак оптимално).
        - Добар избор кога имаш тежински терен (планина) и сакаш
          поконзервативна, секогаш-сигурна проценка.
    """
    return math.sqrt((state_1[0] - state_2[0]) ** 2 + (state_1[1] - state_2[1]) ** 2)


# ---------------------------------------------------------------------------
# 5. Алгоритми за пребарување
# ---------------------------------------------------------------------------
#
# Сите алгоритми се ИМПЛЕМЕНТИРАНИ КАКО ГЕНЕРАТОРИ (користат `yield`).
# Тоа значи дека секој повик на `next(algorithm)` извршува ЕДЕН чекор од
# пребарувањето и враќа tuple: (frontier, current_path, vertex_to_expand).
# Ова е корисно за чекор-по-чекор визуелизација (како во оригиналниот
# нотбук со Plotly/ipywidgets), но истите функции можат лесно да се
# "испуцаат" целосно со `for` јамка ако не ти треба анимација — види
# пример во делот 7 подолу.
# ---------------------------------------------------------------------------

def breadth_first_search(starting_vertex, goal_vertex, expand_state):
    """
    BFS — Пребарување во ширина (Breadth-First Search).

    Структура на податоци: ред (queue, FIFO) — `deque`, со `popleft`
    за вадење и `append` за додавање (секогаш на крајот).

    КОГА да се користи:
        - Кога СИТЕ чекори имаат ЕДНАКВА цена (нетежинска мрежа) и сакаш
          ГАРАНТИРАНО НАЈКРАТКА патека (по број на чекори).
        - BFS е целосен (complete) и оптимален (optimal) кога нема
          тежини — секогаш ја наоѓа најкратката патека.
        - Недостаток: меморијски е поскап бидејќи го памти целиот фронт
          ниво по ниво (особено кај големи/широки мрежи).
        - НЕ користи хеуристика и НЕ користи тежини — целосно
          "наивно" пребарување слој по слој.
    """
    visited = {starting_vertex}
    queue = deque([[starting_vertex]])
    while queue:
        vertex_list = queue.popleft()
        vertex_to_expand = vertex_list[-1]

        frontier = list(set([q[-1] for q in queue]))
        yield frontier, vertex_list, vertex_to_expand

        for neighbour in expand_state(vertex_to_expand):
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(vertex_list + [neighbour])
    yield [], [], goal_vertex


def depth_first_search(starting_vertex, goal_vertex, expand_state):
    """
    DFS — Пребарување во длабочина (Depth-First Search).

    Структура на податоци: стек (стиснат во `deque`) — со `popleft` за
    вадење, НО со `appendleft` за додавање (секогаш на почеток), што го
    претвора редот во стек (LIFO) — оттаму "длабочина" наместо "ширина".

    КОГА да се користи:
        - Кога не ти е важна должината/оптималноста на патеката, туку
          само "да најдеш некоја" патека до целта (или да истражиш цела
          гранка пред да се повлечеш).
        - Меморијски е поефтин од BFS (го памти само патот по една
          гранка наместо цел фронт), но НЕ гарантира најкратка патека.
        - НЕ е оптимален и НЕ користи хеуристика/тежини.
        - Добар кога просторот за пребарување е длабок, а решенијата се
          "длабоко" во дрвото, или кога само треба да провериш
          поврзаност/достапност (дали постои патека, без значење колкава).
    """
    visited = {starting_vertex}
    queue = deque([[starting_vertex]])
    while queue:
        vertex_list = queue.popleft()
        vertex_to_expand = vertex_list[-1]

        frontier = list(set([q[-1] for q in queue]))
        yield frontier, vertex_list, vertex_to_expand

        for neighbour in expand_state(vertex_to_expand):
            if neighbour not in visited:
                visited.add(neighbour)
                queue.appendleft(vertex_list + [neighbour])
    yield [], [], goal_vertex


def uniform_cost_search(starting_vertex, goal_vertex, expand_state, get_weight_fn):
    """
    UCS — Пребарување со униформна цена (Uniform Cost Search) /
    Dijkstra-стил пребарување.

    Структура на податоци: приоритетен ред (`heapq`), подреден по
    збирна тежина на патеката (НЕ по број на чекори).

    КОГА да се користи:
        - Кога чекорите имаат РАЗЛИЧНИ тежини/цени (пр. движење по
          планина каде нагорно движење е поскапо), и сакаш ГАРАНТИРАНО
          најевтина (не нужно најкратка по број чекори!) патека.
        - UCS е целосен и оптимален кога тежините се не-негативни.
        - НЕ користи хеуристика — слепо го шири најевтиниот пат досега
          (без проценка колку е "блиску" до целта), па е поспор од A*
          со добра хеуристика, но секогаш точен.
        - Параметар `get_weight_fn` треба да биде функција
          (state_a, state_b) -> цена_на_чекор, пр. ламбда која ја повикува
          get_weight(a, b, MOUNTAIN).
    """
    expanded = set()
    queue = [(0, [starting_vertex])]
    heapq.heapify(queue)
    while queue:
        weight, vertex_list = heapq.heappop(queue)
        vertex_to_expand = vertex_list[-1]
        if vertex_to_expand in expanded:
            continue

        frontier = list(set([q[-1][-1] for q in queue]))
        yield frontier, vertex_list, vertex_to_expand

        for neighbour in expand_state(vertex_to_expand):
            new_weight = get_weight_fn(vertex_to_expand, neighbour)
            if neighbour not in expanded:
                heapq.heappush(queue, (weight + new_weight, vertex_list + [neighbour]))
        expanded.add(vertex_to_expand)
    yield [], [], goal_vertex


def gready_search(starting_vertex, goal_vertex, expand_state, heuristic):
    """
    Greedy Best-First Search — "лакомо" пребарување.

    Структура на податоци: приоритетен ред подреден ИСКЛУЧИВО по
    вредност на хеуристиката од сосед до целта (игнорира колку е
    "поминато" дотука).

    КОГА да се користи:
        - Кога сакаш брзо решение и не ти е важна оптималноста — само
          сакаш ДОБРА, не нужно најкратка/најевтина патека, најбрзо
          можно.
        - Брз е во пракса (често проширува малку јазли ако хеуристиката
          е добра), но НЕ е оптимален и НЕ е целосен во општ случај
          (може да "залута" во ѓубре локални минимуми или циклуси ако
          хеуристиката не е добра, или ако простор е бесконечен).
        - Зависи целосно од квалитетот на хеуристиката (manhattan_distance,
          chebyshev_distance или eucledian_distance).
        - Добар избор кога меморија/брзина се поважни од гарантирана
          точност (пр. NPC движење во игри, real-time системи).
    """
    expanded = set()
    queue = [(0, [starting_vertex])]
    heapq.heapify(queue)
    while queue:
        weight, vertex_list = heapq.heappop(queue)
        vertex_to_expand = vertex_list[-1]
        if vertex_to_expand in expanded:
            continue

        frontier = list(set([q[-1][-1] for q in queue]))
        yield frontier, vertex_list, vertex_to_expand

        for neighbour in expand_state(vertex_to_expand):
            heuristic_score = heuristic(neighbour, goal_vertex)
            if neighbour not in expanded:
                heapq.heappush(queue, (heuristic_score, vertex_list + [neighbour]))
        expanded.add(vertex_to_expand)
    yield [], [], goal_vertex


def a_star_search(starting_vertex, goal_vertex, expand_state, heuristic, get_weight_fn, alpha=1):
    """
    A* (A-star) пребарување.

    Структура на податоци: приоритетен ред подреден по
        f(n) = g(n) + alpha * h(n)
    каде:
        g(n)     = реалната збирна тежина на патеката од почеток до n
                   (исто како кај UCS),
        h(n)     = хеуристичка проценка од n до целта,
        alpha    = "тежина" на хеуристиката (множител);
                   alpha = 1 → класичен A*,
                   alpha = 0 → се сведува на UCS (бара само по реална цена),
                   alpha > 1 → се приближува кон Greedy (повеќе верува на
                               хеуристиката, побрзо но потенцијално
                               не-оптимално ако h не е admissible).

    КОГА да се користи:
        - Најчесто најдобар избор кога имаш ТЕЖИНСКА мрежа (различни
          цени по чекор) И сакаш и БРЗИНА и ОПТИМАЛНОСТ.
        - Ако хеуристиката `heuristic` е admissible (никогаш не
          преценува реалната преостаната цена) и consistent, A* е
          ГАРАНТИРАНО оптимален (исто како UCS), но обично проширува
          ПОМАЛКУ јазли (побрз) бидејќи хеуристиката го насочува
          пребарувањето кон целта.
        - Избор на хеуристика:
            * expand_state_4  → manhattan_distance (точна за 4-смерно)
            * expand_state_8  → chebyshev_distance (точна за 8-смерно)
                                 или eucledian_distance (поконзервативна)
        - Параметар `get_weight_fn` треба да биде функција
          (state_a, state_b) -> цена_на_чекор.
    """
    expanded = set()
    queue = [((0, 0), [starting_vertex])]
    heapq.heapify(queue)
    while queue:
        weight_tupple, vertex_list = heapq.heappop(queue)
        current_a_star_weight, current_path_weight = weight_tupple
        vertex_to_expand = vertex_list[-1]
        if vertex_to_expand in expanded:
            continue

        frontier = list(set([q[-1][-1] for q in queue]))
        yield frontier, vertex_list, vertex_to_expand

        for neighbour in expand_state(vertex_to_expand):
            if neighbour not in expanded:
                new_weight = get_weight_fn(vertex_to_expand, neighbour)
                heuristic_score = heuristic(neighbour, goal_vertex)
                path_weight = current_path_weight + new_weight
                a_star_weight = path_weight + alpha * heuristic_score
                heapq.heappush(queue, ((a_star_weight, path_weight), vertex_list + [neighbour]))
        expanded.add(vertex_to_expand)
    yield [], [], goal_vertex


# ---------------------------------------------------------------------------
# 6. РЕЗИМЕ — кога која функција да се користи
# ---------------------------------------------------------------------------
"""
┌──────────────────────┬───────────────────────┬────────────┬───────────┬───────────────────────────────┐
│ Алгоритам             │ Структура на податоци │ Хеуристика │ Тежини    │ Кога да се користи             │
├──────────────────────┼───────────────────────┼────────────┼───────────┼───────────────────────────────┤
│ breadth_first_search  │ ред (FIFO)             │ НЕ          │ НЕ        │ Нетежинска мрежа,               │
│                       │                        │            │           │ гарантирано најкратка патека    │
│                       │                        │            │           │ (по број чекори).               │
├──────────────────────┼───────────────────────┼────────────┼───────────┼───────────────────────────────┤
│ depth_first_search    │ стек (LIFO)            │ НЕ          │ НЕ        │ Брзо барање НЕКАКВА патека,     │
│                       │                        │            │           │ помалку меморија, без гаранции. │
├──────────────────────┼───────────────────────┼────────────┼───────────┼───────────────────────────────┤
│ uniform_cost_search   │ приоритетен ред        │ НЕ          │ ДА        │ Тежинска мрежа, потребна        │
│                       │ (по g(n))              │            │           │ гарантирано најевтина патека.   │
├──────────────────────┼───────────────────────┼────────────┼───────────┼───────────────────────────────┤
│ gready_search         │ приоритетен ред        │ ДА          │ НЕ        │ Брзо "добро" решение,           │
│                       │ (по h(n))              │            │ (игнорира)│ без гаранција за оптималност.   │
├──────────────────────┼───────────────────────┼────────────┼───────────┼───────────────────────────────┤
│ a_star_search         │ приоритетен ред        │ ДА          │ ДА        │ Најдобар избор кога има         │
│                       │ (по g(n)+α·h(n))       │            │           │ тежини И сакаш брзина+оптимал.  │
└──────────────────────┴───────────────────────┴────────────┴───────────┴───────────────────────────────┘

Избор на функција за проширување (соседи):
    - expand_state_4  → движење само горе/долу/лево/десно
    - expand_state_8  → движење + дијагонали

Избор на хеуристика (само за gready_search и a_star_search):
    - manhattan_distance  → паруј со expand_state_4
    - chebyshev_distance  → паруј со expand_state_8 (точна на 8-мрежа)
    - eucledian_distance  → паруј со expand_state_8 кога сакаш
                             поконзервативна ("права линија") проценка,
                             особено корисна со тежински терен (планина)
"""


# ---------------------------------------------------------------------------
# 7. Пример за употреба
# ---------------------------------------------------------------------------

def run_algorithm_to_completion(algorithm_generator):
    """
    Помошна функција која "испуцува" целиот генератор-алгоритам без
    чекор-по-чекор анимација и враќа само финалната (последна) патека.

    КОГА да се користи:
        - Кога не ти треба визуелизација/анимација (без Plotly/ipywidgets),
          а само сакаш краен резултат — патеката од старт до цел.
    """
    last_path = []
    for frontier, current_path, vertex_to_expand in algorithm_generator:
        if current_path:
            last_path = current_path
    return last_path


if __name__ == "__main__":
    start = (4, 4)
    goal = (10, 10)

    MOUNTAIN, _ = create_mountain()

    print("=== BFS (4-смерно, без тежини) ===")
    algo = breadth_first_search(start, goal, expand_state=expand_state_4)
    print("Патека:", run_algorithm_to_completion(algo))

    print("\n=== DFS (8-смерно, без тежини) ===")
    algo = depth_first_search(start, goal, expand_state=expand_state_8)
    print("Патека:", run_algorithm_to_completion(algo))

    print("\n=== UCS (8-смерно, со тежини од планината) ===")
    algo = uniform_cost_search(
        start, goal, expand_state=expand_state_8,
        get_weight_fn=lambda a, b: get_weight(a, b, MOUNTAIN),
    )
    print("Патека:", run_algorithm_to_completion(algo))

    print("\n=== Greedy (8-смерно, хеуристика: Евклид) ===")
    algo = gready_search(start, goal, expand_state=expand_state_8, heuristic=eucledian_distance)
    print("Патека:", run_algorithm_to_completion(algo))

    print("\n=== A* (8-смерно, хеуристика: Чебишев, со тежини) ===")
    algo = a_star_search(
        start, goal, expand_state=expand_state_8,
        heuristic=chebyshev_distance,
        get_weight_fn=lambda a, b: get_weight(a, b, MOUNTAIN),
        alpha=1,
    )
    print("Патека:", run_algorithm_to_completion(algo))
